# Tournament Manager — Domain Scenarios

This notebook shows end-to-end domain workflows:
1. Registrar un competidor con ficha médica
2. Inscribir competidores en categorías
3. Aprobar inscripciones
4. Crear un torneo con tatamis
5. Registrar combates y consultar resultados
6. Estadísticas de participación

## 0. Setup

In [ ]:
import os, sys
from datetime import datetime, timedelta

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.environ["APP_ENV"] = "test"
os.environ["DATABASE_URL_TEST"] = "sqlite:///:memory:"

from sqlalchemy import create_engine, event
from sqlalchemy.orm import sessionmaker
from src.domain.models import Base
from src.domain.models import (
    Fighter, Tournament, Category,
    FighterCategoryRegistration, Match
)
from src.domain.repositories import UnitOfWork

engine = create_engine("sqlite:///:memory:", connect_args={"check_same_thread": False})

@event.listens_for(engine, "connect")
def set_fk(dbapi_conn, _):
    dbapi_conn.cursor().execute("PRAGMA foreign_keys=ON")

Base.metadata.create_all(engine)
Session = sessionmaker(bind=engine, autoflush=False, autocommit=False)

def uow():
    return UnitOfWork(session_factory=Session)

print("✅ Setup complete")

## 1. Crear categorías base

In [ ]:
with uow() as u:
    cats = [
        Category(name="Blue -57 kg M",  belt_level="blue",  weight_min_kg=0,  weight_max_kg=57,  gender="M"),
        Category(name="Blue -70 kg F",  belt_level="blue",  weight_min_kg=57, weight_max_kg=70,  gender="F"),
        Category(name="Black -70 kg M", belt_level="black", weight_min_kg=0,  weight_max_kg=70,  gender="M"),
    ]
    for c in cats:
        u.categories.add(c)
    u.commit()
    cids = [c.category_id for c in cats]

print("Categorías creadas:", cids)

## 2. Crear torneo con tatamis

In [ ]:
now = datetime.utcnow()

with uow() as u:
    torneo = Tournament(
        name="Copa Primavera 2026",
        location="Palau Sant Jordi, Barcelona",
        start_date=now + timedelta(days=30),
        end_date=now + timedelta(days=31),
        max_fighters=64,
    )
    u.tournaments.add(torneo)
    u.commit()
    tid = torneo.tournament_id

    # Añadir tatamis
    for i in range(1, 4):
        u.tournaments.add_tatami_to_tournament(
            tournament_id=tid,
            name=f"Tatami {chr(64+i)}",
            mat_number=i,
            area_size=64.0,
        )
    u.commit()

print(f"Torneo '{torneo.name}' creado con ID={tid}")

with uow() as u:
    t = u.tournaments.get(tid)
    print(f"Tatamis asignados: {[ta.name for ta in t.tatamis]}")

## 3. Registrar competidores con ficha médica

In [ ]:
fighters_data = [
    dict(first_name="Ana",     last_name="García",   gender="F", belt_level="blue",  country="Spain", email="ana@ex.com"),
    dict(first_name="Carlos",  last_name="López",    gender="M", belt_level="black", country="Spain", email="carlos@ex.com"),
    dict(first_name="Yuki",    last_name="Tanaka",   gender="F", belt_level="blue",  country="Japan", email="yuki@ex.com"),
    dict(first_name="Mohamed", last_name="Ali",      gender="M", belt_level="blue",  country="Egypt", email="moh@ex.com"),
]

fids = []
with uow() as u:
    for fd in fighters_data:
        f = Fighter(**fd)
        u.fighters.add(f)
    u.commit()
    fids = [f.fighter_id for f in u.fighters.get_all()]

print(f"{len(fids)} competidores registrados: IDs {fids}")

# Añadir fichas médicas
medicals = [
    dict(fighter_id=fids[0], blood_type="A+",  doctor_name="Dr. Pérez",    notes="Sin patologías"),
    dict(fighter_id=fids[1], blood_type="O-",  doctor_name="Dr. Ruiz",     notes="Asma leve"),
    dict(fighter_id=fids[2], blood_type="B+",  doctor_name="Dr. Yamamoto", notes="Sin patologías"),
    dict(fighter_id=fids[3], blood_type="AB+", doctor_name="Dr. Karim",    notes="Sin patologías"),
]

with uow() as u:
    for m in medicals:
        u.fighters.add_medical_record(**m)
    u.commit()

print("Fichas médicas añadidas")

## 4. Inscribir competidores en categorías

In [ ]:
inscripciones = [
    # (fighter_idx, category_idx, peso_pesaje)
    (0, 1, 56.2),   
    (1, 2, 68.5), 
    (2, 1, 65.0), 
    (3, 0, 56.8), 
]

with uow() as u:
    for fi, ci, peso in inscripciones:
        reg = FighterCategoryRegistration(
            fighter_id=fids[fi],
            category_id=cids[ci],
            tournament_id=tid,
            weight_in_weight_kg=peso,
            is_approved=False,
        )
        u.registrations.add(reg)
    u.commit()

with uow() as u:
    pending = u.registrations.get_pending_approvals(tid)
    print(f"{len(pending)} inscripciones pendientes de aprobación")

## 5. Aprobar inscripciones

In [ ]:
with uow() as u:
    # Aprobar todas menos la de Yuki (simular: no pasó el pesaje)
    to_approve = [
        (fids[0], cids[1]),
        (fids[1], cids[2]),
        (fids[3], cids[0]),
    ]
    for fid, cid in to_approve:
        u.fighters.approve_registration(
            fighter_id=fid, category_id=cid, tournament_id=tid
        )
    u.commit()

with uow() as u:
    all_regs = u.registrations.get_by_tournament(tid)
    approved  = [r for r in all_regs if r.is_approved]
    pending   = [r for r in all_regs if not r.is_approved]
    print(f"Aprobadas: {len(approved)} | Pendientes: {len(pending)}")

## 6. Registrar combates

In [ ]:
with uow() as u:
    match1 = Match(
        tournament_id=tid,
        category_id=cids[1],
        blue_fighter_id=fids[0],   
        red_fighter_id=fids[2],   
        winner_id=fids[0],      
        round_number=1,
        duration_seconds=185.0,
        win_method="ippon",
        scheduled_time=now + timedelta(days=30, hours=9),
    )
    match2 = Match(
        tournament_id=tid,
        category_id=cids[2],
        blue_fighter_id=fids[1],  
        red_fighter_id=fids[3],   
        winner_id=fids[1],       
        round_number=1,
        duration_seconds=245.0,
        win_method="waza-ari",
        scheduled_time=now + timedelta(days=30, hours=10),
    )
    u.matches.add(match1)
    u.matches.add(match2)
    u.commit()
    mids = [match1.match_id, match2.match_id]

print(f"Combates registrados: {mids}")

## 7. Consultar resultados y estadísticas

In [ ]:
with uow() as u:
    print("── Combates del torneo ──────────────────────")
    for m in u.matches.get_by_tournament(tid):
        print(f"  Ronda {m.round_number}: método={m.win_method}, duración={m.duration_seconds}s")

    print("\n── Victorias de Ana García ──────────────────")
    for m in u.matches.get_wins_by_fighter(fids[0]):
        print(f"  Match {m.match_id}: ganó por {m.win_method}")

    print("\n── Paginación de competidores (página 1 de 2) ──")
    page = u.fighters.get_paginated(page=1, page_size=2)
    for f in page:
        print(f"  {f.last_name}, {f.first_name} — {f.belt_level} / {f.country}")

    print(f"\n── Total competidores: {u.fighters.count()} ──")
    print(f"── Torneos activos: {len(u.tournaments.get_active_tournaments())} ──")

## 8. Estadísticas de participación por categoría

In [ ]:
with uow() as u:
    print("── Inscritos por categoría ──────────────────")
    for cat in u.categories.get_all():
        regs = [r for r in u.registrations.get_all() if r.category_id == cat.category_id]
        print(f"  {cat.name}: {len(regs)} inscrito(s)")

---
## ✅ Todos los escenarios de dominio ejecutados correctamente

In [ ]:
print("=" * 55)
print("ESCENARIOS DE DOMINIO COMPLETADOS EXITOSAMENTE ✅")
print("=" * 55)